# helpers

> Helpers to transform Pandas time-series data into baseline ECharts options dictionaries, while still allowing user customisation.

In [ ]:
#| default_exp helpers

In [ ]:
#| export
from __future__ import annotations
from typing import TYPE_CHECKING
if TYPE_CHECKING: import pandas as pd

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import pandas as pd
import numpy as np

from fh_echarts.core import EChart, preview_echart

In [ ]:
#| export
def ts_options(data: pd.DataFrame | pd.Series, x: str = None, y: list | str = None, kind: str = "line") -> dict:
    """
    Transforms a Pandas DataFrame or Series into a baseline ECharts time-series options dictionary.
    """
    import pandas as pd

    # 1. Handle Pandas Series gracefully
    if isinstance(data, pd.Series):
        # Give it a default name if it lacks one, so it can be used as a y-column
        name = data.name if data.name is not None else "value"
        plot_df = data.to_frame(name=name)
    else:
        plot_df = data.copy()

    # 2. Handle the X-axis (Time)
    if x is None:
        # Fall back to the index if no x column is provided
        x = plot_df.index.name or "date"
        plot_df.index.name = x
        plot_df = plot_df.reset_index()
    
    # Format datetimes safely to ISO 8601 while PRESERVING timezone offsets
    if pd.api.types.is_datetime64_any_dtype(plot_df[x]):
        plot_df[x] = plot_df[x].apply(lambda dt: dt.isoformat() if pd.notnull(dt) else None)

    # 3. Handle the Y-axis (Values)
    if y is None:
        # Default to all numeric columns except the time column
        y = plot_df.select_dtypes(include='number').columns.tolist()
        if x in y: y.remove(x)
    elif isinstance(y, str):
        y = [y]

    # 4. Handle NaNs cleanly (Convert to None for JSON `null`)
    # Converting to object dtype first prevents Pandas from trying to force NaNs back into floats
    plot_df = plot_df[[x] + y].astype(object).where(pd.notna(plot_df), None)

    # 5. Construct the ECharts Dataset
    headers = [x] + y
    dataset_source = [headers] + plot_df.values.tolist()

    # 6. Build the baseline ECharts dictionary
    options = {
        "dataset": {
            "source": dataset_source
        },
        "tooltip": {
            "trigger": "axis"
        },
        "xAxis": {
            "type": "time", 
            "name": str(x)
        },
        "yAxis": {
            "type": "value"
        },
        "series": [
            {
                "type": kind, 
                "name": str(col), 
                "encode": {"x": str(x), "y": str(col)}
            } 
            for col in y
        ]
    }

    return options

## Examples

Create some sample time-series data to demonstrate `ts_options`:

In [ ]:
dates = pd.date_range("2024-01-01", periods=30, freq="D")
df = pd.DataFrame({"date": dates, "temperature": 20 + 5 * np.random.randn(30).cumsum() * 0.1, "humidity": 60 + 3 * np.random.randn(30).cumsum() * 0.1})
df.head()

,date,temperature,humidity
0,2024-01-01,20.147174,59.807882
1,2024-01-02,19.853315,59.886825
2,2024-01-03,19.951529,59.373097
3,2024-01-04,19.591586,59.503999
4,2024-01-05,19.548946,59.759795


### Basic usage — DataFrame with explicit x column

In [ ]:
opts = ts_options(df, x="date")
chart = EChart(opts)
preview_echart(chart)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead


  warnings.warn("Consider using IPython.display.IFrame instead")


### Selecting specific y columns

In [ ]:
opts_single = ts_options(df, x="date", y="temperature")
chart_single = EChart(opts_single)
preview_echart(chart_single)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead


  warnings.warn("Consider using IPython.display.IFrame instead")


### Using a DatetimeIndex (no explicit x column)

In [ ]:
df_idx = df.set_index("date")
opts_idx = ts_options(df_idx)
chart_idx = EChart(opts_idx)
preview_echart(chart_idx)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead


  warnings.warn("Consider using IPython.display.IFrame instead")


### From a Pandas Series

In [ ]:
series = df.set_index("date")["temperature"]
opts_series = ts_options(series)
chart_series = EChart(opts_series)
preview_echart(chart_series)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead


  warnings.warn("Consider using IPython.display.IFrame instead")


### Customising the result

`ts_options` returns a plain dictionary, so you can merge in any ECharts options before passing to `EChart`:

In [ ]:
opts_custom = ts_options(df, x="date")
opts_custom["title"] = {"text": "Weather Station"}
opts_custom["legend"] = {"show": True}
opts_custom["yAxis"]["name"] = "°C / %"

chart_custom = EChart(opts_custom, theme="dark")
preview_echart(chart_custom)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead


  warnings.warn("Consider using IPython.display.IFrame instead")


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()